## v0.1.1 reliability fix

The final smoke-test cell now refreshes ComfyUI node metadata itself. This removes the previous `object_info` cross-cell dependency observed on mobile Colab.

# TRAMA Visual Factory — Q4 Smoke Test v0.1

**Candidate:** MZ-VF-001  
**Adapter:** Google Colab Free / Tesla T4 candidate  
**Purpose:** one bounded 512×512 Qwen-Image-Edit GGUF smoke test.  
**Paid compute authorised:** NO

This notebook installs only open tooling, downloads a Q4 model package, runs one image edit, and writes a machine-readable receipt. It does **not** publish anything.

In [ ]:
import json, os, shutil, subprocess, sys

def sh(cmd):
    return subprocess.check_output(cmd, shell=True, text=True).strip()

if shutil.which('nvidia-smi') is None:
    raise SystemExit('STOP: no NVIDIA GPU. Re-enable the free GPU runtime; do not download models.')

gpu_name = sh("nvidia-smi --query-gpu=name --format=csv,noheader | head -1")
vram_mib = float(sh("nvidia-smi --query-gpu=memory.total --format=csv,noheader,nounits | head -1"))
vram_gib = vram_mib / 1024
ram_kib = int(sh("awk '/MemTotal/ {print $2}' /proc/meminfo"))
ram_gib = ram_kib / 1024 / 1024
disk = shutil.disk_usage('/content')
disk_free_gib = disk.free / (1024**3)

print({'gpu': gpu_name, 'vramGiB': round(vram_gib,2), 'ramGiB': round(ram_gib,2), 'diskFreeGiB': round(disk_free_gib,2)})

if vram_gib < 14 or ram_gib < 11 or disk_free_gib < 35:
    raise SystemExit('STOP: runtime no longer matches the T4_Q4_CANDIDATE envelope.')

print('T4_Q4_CANDIDATE confirmed. Proceeding to one bounded smoke test.')

## Install ComfyUI + GGUF support

The notebook uses the official ComfyUI repository and the city96 ComfyUI-GGUF loader.

In [ ]:
import os, subprocess, sys

COMFY='/content/ComfyUI'
if not os.path.exists(COMFY):
    subprocess.run(['git','clone','--depth','1','https://github.com/Comfy-Org/ComfyUI.git',COMFY], check=True)

subprocess.run([sys.executable,'-m','pip','install','-q','-r',f'{COMFY}/requirements.txt'], check=True)

GGUF=f'{COMFY}/custom_nodes/ComfyUI-GGUF'
if not os.path.exists(GGUF):
    subprocess.run(['git','clone','--depth','1','https://github.com/city96/ComfyUI-GGUF.git',GGUF], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-r',f'{GGUF}/requirements.txt'], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','huggingface_hub>=0.27','requests','pillow'], check=True)

print('Install complete.')

## Download pinned Q4 smoke-test package

Downloads are public/open model artifacts. Expect a large download (roughly tens of GB total). The notebook records SHA-256 after download.

In [ ]:
from huggingface_hub import hf_hub_download
from pathlib import Path
import hashlib, os, json

COMFY=Path('/content/ComfyUI')
targets = {
    'unet': COMFY/'models'/'unet',
    'text': COMFY/'models'/'text_encoders',
    'vae': COMFY/'models'/'vae',
    'lora': COMFY/'models'/'loras',
}
for p in targets.values(): p.mkdir(parents=True, exist_ok=True)

specs = [
    ('Aitrepreneur/FLX','Qwen_Image_Edit-Q4_K_S.gguf',targets['unet']),
    ('Aitrepreneur/FLX','Qwen2.5-VL-7B-Instruct-UD-Q4_K_S.gguf',targets['text']),
    ('Aitrepreneur/FLX','Qwen2.5-VL-7B-Instruct-mmproj-BF16.gguf',targets['text']),
    ('Aitrepreneur/FLX','qwen_image_vae.safetensors',targets['vae']),
    ('Aitrepreneur/FLX','Qwen-Image-Lightning-4steps-V1.0.safetensors',targets['lora']),
]

downloaded=[]
for repo_id, filename, folder in specs:
    path=hf_hub_download(repo_id=repo_id, filename=filename, local_dir=str(folder))
    downloaded.append(path)
    print('ready:', path)

def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(1024*1024), b''):
            h.update(chunk)
    return h.hexdigest()

model_receipt={os.path.basename(p):{'bytes':os.path.getsize(p),'sha256':sha256(p)} for p in downloaded}
print(json.dumps(model_receipt, indent=2))

## Create a neutral 512×512 test plate

This is not a MUSEO ZERO asset. It only proves that the Q4 edit pipeline can produce an image on the free T4.

In [ ]:
from PIL import Image, ImageDraw
from pathlib import Path

img=Image.new('RGB',(512,512),(18,25,36))
d=ImageDraw.Draw(img)
d.rectangle((40,70,472,450), fill=(34,45,60), outline=(120,140,160), width=3)
d.rectangle((160,130,390,300), fill=(25,70,90), outline=(80,190,210), width=4)
d.rectangle((65,340,445,370), fill=(70,85,100))
d.ellipse((75,310,105,340), fill=(230,150,40))
input_path=Path('/content/ComfyUI/input/trama_q4_smoke.png')
input_path.parent.mkdir(parents=True, exist_ok=True)
img.save(input_path)
display(img)

## Start ComfyUI in low-VRAM mode and verify required nodes

In [ ]:
import subprocess, sys, time, requests, os, signal

COMFY='/content/ComfyUI'
LOG='/content/comfyui-q4-smoke.log'
logf=open(LOG,'w')
proc=subprocess.Popen([
    sys.executable,'main.py','--lowvram','--reserve-vram','0.8',
    '--listen','127.0.0.1','--port','8188','--disable-api-nodes'
], cwd=COMFY, stdout=logf, stderr=subprocess.STDOUT)

base='http://127.0.0.1:8188'
for _ in range(120):
    try:
        r=requests.get(base+'/object_info',timeout=2)
        if r.ok:
            object_info=r.json(); break
    except Exception:
        pass
    time.sleep(2)
else:
    logf.flush()
    print(open(LOG).read()[-5000:])
    raise SystemExit('SMOKE_FAIL_STARTUP')

required=['LoadImage','VAELoader','UnetLoaderGGUF','TextEncodeQwenImageEditPlus','VAEEncode','KSampler','VAEDecode','SaveImage']
missing=[n for n in required if n not in object_info]
if missing:
    print('Missing nodes:',missing)
    raise SystemExit('SMOKE_FAIL_NODES')

clip_node='CLIPLoaderGGUF' if 'CLIPLoaderGGUF' in object_info else None
if not clip_node:
    raise SystemExit('SMOKE_FAIL_NODES: CLIPLoaderGGUF missing')

print('Required nodes present. Using',clip_node)

## Queue one bounded edit

Prompt: convert the neutral blockout into a restrained contemporary museum interior. No text, no labels, no people.

In [ ]:
import requests, time, json, os, traceback
from PIL import Image
from IPython.display import display

base='http://127.0.0.1:8188'

# Refresh ComfyUI node metadata here so this cell is self-contained.
r=requests.get(base+'/object_info', timeout=10)
r.raise_for_status()
object_info=r.json()
required=['LoadImage','VAELoader','UnetLoaderGGUF','TextEncodeQwenImageEditPlus','VAEEncode','KSampler','VAEDecode','SaveImage','CLIPLoaderGGUF']
missing=[n for n in required if n not in object_info]
if missing:
    raise SystemExit(f'SMOKE_FAIL_NODES: missing {missing}')

clip_inputs={'clip_name':'Qwen2.5-VL-7B-Instruct-UD-Q4_K_S.gguf','type':'qwen_image'}
clip_required=object_info['CLIPLoaderGGUF']['input']['required']
if 'device' in clip_required: clip_inputs['device']='default'

def encode_inputs(prompt):
    x={'clip':['2',0],'prompt':prompt,'vae':['3',0]}
    req=object_info['TextEncodeQwenImageEditPlus']['input']['required']
    for key in ('image1','image2','image3'):
        if key in req: x[key]=['1',0]
    return x

workflow={
 '1':{'class_type':'LoadImage','inputs':{'image':'trama_q4_smoke.png'}},
 '2':{'class_type':'CLIPLoaderGGUF','inputs':clip_inputs},
 '3':{'class_type':'VAELoader','inputs':{'vae_name':'qwen_image_vae.safetensors'}},
 '4':{'class_type':'UnetLoaderGGUF','inputs':{'unet_name':'Qwen_Image_Edit-Q4_K_S.gguf'}},
 '13':{'class_type':'LoraLoaderModelOnly','inputs':{'model':['4',0],'lora_name':'Qwen-Image-Lightning-4steps-V1.0.safetensors','strength_model':1.0}},
 '5':{'class_type':'TextEncodeQwenImageEditPlus','inputs':encode_inputs('Transform this simple blockout into a polished contemporary museum interior after closing, cinematic editorial illustration, restrained warm amber and cool blue lighting, clean architecture, realistic exhibition materials, no text, no labels, no watermark, no people.')},
 '6':{'class_type':'TextEncodeQwenImageEditPlus','inputs':encode_inputs('text, labels, watermark, poster, infographic, dashboard, people, distorted geometry')},
 '7':{'class_type':'VAEEncode','inputs':{'pixels':['1',0],'vae':['3',0]}},
 '8':{'class_type':'KSampler','inputs':{'model':['13',0],'seed':20261004,'steps':4,'cfg':1.0,'sampler_name':'euler','scheduler':'simple','positive':['5',0],'negative':['6',0],'latent_image':['7',0],'denoise':1.0}},
 '11':{'class_type':'VAEDecode','inputs':{'samples':['8',0],'vae':['3',0]}},
 '12':{'class_type':'SaveImage','inputs':{'images':['11',0],'filename_prefix':'TRAMA_Q4_SMOKE'}}
}

receipt={'schemaVersion':'trama.visual-factory.q4-smoke/v0.1','candidateId':'MZ-VF-001','adapter':'google-colab-free','gpuName':gpu_name,'vramGiB':round(vram_gib,2),'paidComputeAuthorized':False,'decision':'UNKNOWN'}

try:
    q=requests.post(base+'/prompt',json={'prompt':workflow},timeout=30)
    q.raise_for_status()
    prompt_id=q.json()['prompt_id']
    print('queued:',prompt_id)
    result=None
    for _ in range(900):
        h=requests.get(base+f'/history/{prompt_id}',timeout=10).json()
        if prompt_id in h:
            result=h[prompt_id]
            status=result.get('status',{})
            if status.get('completed') or result.get('outputs'):
                break
        time.sleep(2)
    if not result or not result.get('outputs'):
        raise RuntimeError('No output returned before timeout/completion.')
    images=result['outputs'].get('12',{}).get('images',[])
    if not images:
        raise RuntimeError('SaveImage produced no image metadata.')
    meta=images[0]
    out_path=os.path.join('/content/ComfyUI/output',meta.get('subfolder',''),meta['filename'])
    display(Image.open(out_path))
    receipt.update({'decision':'Q4_SMOKE_PASS','promptId':prompt_id,'outputFile':meta['filename'],'outputPath':out_path})
except Exception as e:
    logf.flush()
    tail=open(LOG,errors='ignore').read()[-12000:]
    lower=(str(e)+'\n'+tail).lower()
    if 'out of memory' in lower or 'cuda oom' in lower:
        receipt['decision']='Q4_OOM_FALLBACK_Q3_ALLOWED'
    else:
        receipt['decision']='Q4_SMOKE_FAIL_DIAGNOSE'
    receipt['error']=str(e)
    print(tail[-5000:])

with open('/content/trama-q4-smoke-receipt.json','w',encoding='utf-8') as f:
    json.dump(receipt,f,indent=2)
print(json.dumps(receipt,indent=2))

## Interpretation

- `Q4_SMOKE_PASS` → free T4 backend is technically qualified for the next bounded MUSEO ZERO production test.
- `Q4_OOM_FALLBACK_Q3_ALLOWED` → one controlled Q3 fallback is allowed; do not retry Q4 repeatedly.
- `Q4_SMOKE_FAIL_DIAGNOSE` → stop and inspect the specific error; do not sample/retry blindly.